# CAT-Seg segmentation (blockology-gvi)

Runs the same `setup_catseg.py` / `run_inference.py` from [`segmentation/cat-seg`](https://github.com/jling888/blockology-gvi/tree/main/segmentation/cat-seg) directly on this Colab runtime -- no local `colab` CLI required.

**Before running:** `Runtime > Change runtime type > T4/L4 GPU`.

**Setup (once):** upload `output/imagery/` (its `raw/` subfolder + `raw_manifest.csv` sibling) to your Google Drive, e.g. `MyDrive/blockology-gvi/imagery/`. No checkpoint download needed -- `setup_catseg.py` fetches it to Drive on first run.

Masks/overlays/`pixel_counts.csv` are written straight to the mounted Drive `out_dir` -- since this notebook *is* the Colab session, there's no separate download step (unlike `run_on_colab.sh`, which drives a remote session and has to pull files back over the `colab` CLI).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@markdown Paths are relative to Drive's mounted root (`/content/drive/`).
imagery_dir = "MyDrive/blockology-gvi/imagery"  #@param {type:"string"}
checkpoint_dir = "MyDrive/blockology-gvi"  #@param {type:"string"}
out_dir = "MyDrive/blockology-gvi/catseg_out"  #@param {type:"string"}
#@markdown Repo/ref to pull `setup_catseg.py` / `run_inference.py` / `vocabulary.json` from.
repo_url = "https://github.com/jling888/blockology-gvi.git"  #@param {type:"string"}
repo_ref = "main"  #@param {type:"string"}

images_dir = f"/content/drive/{imagery_dir}/raw"
manifest_path = f"/content/drive/{imagery_dir}/raw_manifest.csv"
checkpoint_path = f"/content/drive/{checkpoint_dir}/model_large.pth"
drive_out_dir = f"/content/drive/{out_dir}"

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU -- set Runtime > Change runtime type > GPU before continuing."

In [ ]:
![ -d /content/blockology-gvi ] || git clone --depth 1 --branch "$repo_ref" "$repo_url" /content/blockology-gvi
%cd /content/blockology-gvi/segmentation/cat-seg

In [ ]:
# Clones CAT-Seg, installs detectron2 + deps, downloads the ViT-L/14 checkpoint to Drive if absent.
!python setup_catseg.py --checkpoint-dir "{checkpoint_dir if checkpoint_dir.startswith('/') else '/content/drive/' + checkpoint_dir}"

In [ ]:
# Segments every raw/*.jpg, writes masks/overlays/pixel_counts.csv straight to Drive.
# Resumable: skips images already in pixel_counts.csv (add --force to redo).
!python run_inference.py \
  --checkpoint "{checkpoint_path}" \
  --vocab vocabulary.json \
  --images-dir "{images_dir}" \
  --out-dir "{drive_out_dir}" \
  --manifest "{manifest_path}"

Output on Drive at `out_dir`: `pixel_counts.csv`, `masks/*.npz`, `overlays/*.png` -- feed `pixel_counts.csv` to `cli.py --stage metrics` (see `segmentation/cat-seg/README.md`).